In [3]:
import torch
import torch.nn.utils.prune as prune
from ultralytics import YOLO

# Function to prune Conv2D layers (structured pruning for better performance)
def prune_model(model, amount=0.1):
    for module in model.modules():
        if isinstance(module, torch.nn.Conv2d):
            prune.ln_structured(module, name='weight', amount=amount, n=2, dim=0)  # Channel pruning
            prune.remove(module, 'weight')  # Make pruning permanent
    return model
weight_path = '/root/autodl-tmp/STN-YOLO/antenna/fine_tuning/antenna/YOLOV8N/Run_15/weights/best.pt'
data_path = "/root/autodl-tmp/STN-YOLO/antennasets/data.yaml"
prune_path = 'yolov8n_Structured_pruned_weights.pth'
# Load YOLO model
model = YOLO(weight_path)

In [4]:
# Validate the original model
results_str = model.val(data=data_path)
print(f"Original mAP50-95: {results_str.box.map}")

# Access the PyTorch model
torch_model_structured = model.model

# Apply pruning
print("Pruning model...")
pruned_torch_model_structured = prune_model(torch_model_structured, amount=0.1)
print("Model pruned.")

# Save pruned model
torch.save(pruned_torch_model_structured.state_dict(), prune_path)
print("Pruned model weights saved as 'yolov8n_Structured_pruned_weights.pth'.")

Ultralytics YOLOv8.0.120 🚀 Python-3.12.3 torch-2.3.0+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24135MiB)
Model summary (fused): 168 layers, 3006233 parameters, 0 gradients, 8.1 GFLOPs
val: Scanning /root/autodl-tmp/STN-YOLO/antennasets/valid/labels.cache... 390 images, 0 backgrounds, 0 corrupt: 100%|██████████| 390/390 [00:00<?, ?it/s]
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 112, len(boxes) = 614. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 25/25 [00:04<00:00,  5.99it/s]
                   all        390        614      0.924      0.902      0.936      0.704
           GSM Antenna        390        194      0.946      0.905      0.935      0.726
     Microwave Antenna        390        119      0.925      0.934      0.958

Original mAP50-95: 0.7036421689643414
Pruning model...
Model pruned.
Pruned model weights saved as 'yolov8n_Structured_pruned_weights.pth'.


In [ ]:
# Reload pruned weights into a YOLO model
pruned_model = YOLO(weight_path)  # Load the original YOLO model
pruned_model.model.load_state_dict(torch.load(prune_path), strict=False)

# Validate pruned model
results_str = pruned_model.val(data=data_path)
print(f"Pruned mAP50-95: {results_str.box.map}")

# Fine-tune pruned model
print("Fine-tuning pruned model...")
results_str = pruned_model.train(
    data=data_path,
    epochs=5,
    imgsz=640,
    batch=64,
    lr0=0.0001  # Lower learning rate for stability
)


In [9]:
prune_finetune_path = 'yolov8n_structured_pruned_finetuned.pt'
torch.save(pruned_model.state_dict(), prune_finetune_path)
print("Pruned and fine-tuned model saved as 'yolov8n_structured_pruned_finetuned.pt'.")

# Validate fine-tuned model
fine_tuned_model = YOLO(prune_finetune_path)
results_str = fine_tuned_model.val(data=data_path)
print(f"Fine-tuned mAP50-95: {results_str.box.map}")

Pruned and fine-tuned model saved as 'yolov8n_structured_pruned_finetuned.pt'.


KeyError: 'model'